##### 코랩을 사용할 경우

In [ ]:
try:
    # Google Drive를 Colab(/content/drive)rmfja 에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec05
    %ls
except ImportError:
    pass

##### 이전 노트북 실행

In [ ]:
%%capture
get_ipython().run_line_magic("run", "01_data_preprocessing.ipynb")
get_ipython().run_line_magic("run", "02_model_definition.ipynb")

##### 임포트

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

##### Device 설정

In [4]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cuda


##### 옵티마이저 생성

In [5]:
# 손실 함수(CrossEntropyLoss) 생성
loss_fn = nn.CrossEntropyLoss()

# Adam 옵티마이저(Optimizer) 생성
# lr=0.001: 학습률 (모든 파라미터(가중치, 편향)를 얼마나 크게 변경할지)
optimizer = optim.Adam(model.parameters(), lr=0.001)

##### 학습하기 (에포크마다 검증 포함)

In [6]:
# 학습 에포크 및 배치 크기 설정
epochs    = 10
batch_size = 16

# 에포크 반복
for epoch in range(epochs):
    # 훈련 하기 -----------------------------------------------------
    # 학습 모드로 설정: 파라미터 업데이트 및 드롭아웃 활성화
    model.train()

    # 훈련 손실과 정확도 누적 변수 초기화
    batch_loss_sum   = 0
    batch_correct_sum = 0

    for i in range(0, len(train_x_tensor), batch_size):
        batch_x_tensor = train_x_tensor[i:i + batch_size]
        batch_y_tensor = train_y_tensor[i:i + batch_size]

        # 1. 그래디언트 초기화
        optimizer.zero_grad()

        # 2. 모델 출력 얻기
        batch_train_outputs = model(batch_x_tensor)

        # 3. 손실 계산
        loss = loss_fn(batch_train_outputs, batch_y_tensor)

        # 4. 역전파: 그래디언트 계산
        loss.backward()

        # 5. 가중치 업데이트
        optimizer.step()

        # 각 배치 손실의 합산
        batch_loss_sum  += loss.item()

        # 각 배치에서 맞춘 개수 합산: sum(): True(=1) 개수 합산
        batch_correct_sum += (batch_train_outputs.argmax(dim=1) == batch_y_tensor).sum().item()

    # 훈련 평균 손실: 각 배치 손실의 합산 / 배치 개수
    epoch_loss = batch_loss_sum / (len(train_x_tensor) / batch_size)

    # 훈련 정확도: 맞춘 개수 / 전체 개수 * 100
    epoch_acc = batch_correct_sum / len(train_x_tensor) * 100

    # 검증 하기 -----------------------------------------------------
    # 평가 모드로 설정: 파라미터 업데이트 및 드롭아웃 비활성화
    model.eval()

    # 검증 손실과 정확도 누적 변수 초기화
    val_loss    = 0
    val_correct = 0

    # 역전파/기울기 계산없이 검증셋으로 수행
    with torch.inference_mode():  # torch.no_grad():
        # 모델 출력 얻기
        val_outputs = model(val_x_tensor)
        # 검증 손실
        val_loss = loss_fn(val_outputs, val_y_tensor).item()
        # 검증 정확도
        val_correct = (val_outputs.argmax(dim=1) == val_y_tensor).sum().item()
        val_acc = val_correct / len(val_x_tensor) * 100

    # 출력하기 -----------------------------------------------------
    # 에포크마다 훈련·검증 지표를 나란히 출력하여 과적합 여부 확인
    print(f"Epoch [{epoch+1:3d}/{epochs}]  "
            f"훈련 손실: {epoch_loss:.4f}  훈련 정확도: {epoch_acc:5.1f}%  "
            f"검증 손실: {val_loss:.4f}  검증 정확도: {val_acc:5.1f}%")

Epoch [  1/10]  훈련 손실: 1.1189  훈련 정확도:  30.3%  검증 손실: 1.0623  검증 정확도:  38.9%
Epoch [  2/10]  훈련 손실: 1.0537  훈련 정확도:  48.6%  검증 손실: 0.9950  검증 정확도:  77.8%
Epoch [  3/10]  훈련 손실: 0.9729  훈련 정확도:  76.8%  검증 손실: 0.9184  검증 정확도: 100.0%
Epoch [  4/10]  훈련 손실: 0.8916  훈련 정확도:  87.3%  검증 손실: 0.8227  검증 정확도:  94.4%
Epoch [  5/10]  훈련 손실: 0.7943  훈련 정확도:  88.0%  검증 손실: 0.7088  검증 정확도:  94.4%
Epoch [  6/10]  훈련 손실: 0.6382  훈련 정확도:  93.7%  검증 손실: 0.5812  검증 정확도:  94.4%
Epoch [  7/10]  훈련 손실: 0.5261  훈련 정확도:  95.1%  검증 손실: 0.4513  검증 정확도: 100.0%
Epoch [  8/10]  훈련 손실: 0.4112  훈련 정확도:  96.5%  검증 손실: 0.3352  검증 정확도: 100.0%
Epoch [  9/10]  훈련 손실: 0.3275  훈련 정확도:  96.5%  검증 손실: 0.2462  검증 정확도: 100.0%
Epoch [ 10/10]  훈련 손실: 0.2508  훈련 정확도:  97.2%  검증 손실: 0.1827  검증 정확도: 100.0%
